<a href="https://colab.research.google.com/github/joudahmedcodes/Customer-Service-chatbot/blob/main/Joud_Ahmed_Rule_Based_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# General Types of Chatbots

Chatbots can be categorized based on their functionality, intelligence, and interaction style. Here are the main types, along with context for your students:

---

## 1. Rule-Based Chatbots

**Description:** Use predefined rules and patterns (e.g., keyword matching or decision trees) to respond.  
**Example:** A chatbot that replies “Book now!” if the user types “book a flight.”  
**Pros:** Simple, predictable, easy to implement.  
**Cons:** Limited flexibility, struggles with varied inputs.  


---

## 2. ML-Driven Chatbots

**Description:** Use machine learning to understand and classify user input, such as intent detection or sentiment analysis.  
**Example:** A customer service bot that classifies queries as “refund” or “support” using logistic regression.  
**Pros:** Handles diverse inputs better than rule-based systems.  
**Cons:** Requires labeled training data and preprocessing.  

---

## 3. Generative Chatbots

**Description:** Use generative AI (e.g., GPT or BERT) to create dynamic, context-aware responses.  
**Example:** ChatGPT, which generates human-like replies without predefined responses.  
**Pros:** Highly conversational, adaptable to varied inputs.  
**Cons:** Complex, resource-intensive, and requires advanced NLP models.  


---

## 4. Hybrid Chatbots

**Description:** Combine rule-based and ML/generative approaches for greater flexibility and reliability.  
**Example:** A bot that uses ML to classify intents and falls back to rules for specific commands.  
**Pros:** Balances simplicity and sophistication.  
**Cons:** More complex to design and maintain.  


---

## 5. Task-Oriented Chatbots

**Description:** Focus on completing specific tasks like booking, scheduling, or support.  
**Example:** A flight booking bot that walks users through the booking steps.  
**Pros:** Efficient and effective in narrow domains.  
**Cons:** Limited to the tasks they’re designed for.  


---

## 6. Conversational or Virtual Assistant Chatbots

**Description:** Designed for general conversation and assistance, often with personality (e.g., Siri, Alexa).  
**Example:** A bot answering weather questions, greetings, or general FAQs.  
**Pros:** Engaging and widely applicable.  
**Cons:** Requires broad intent recognition and strong NLP capabilities.  


### Summary of Types (for quick reference)

| Type                    | Core Idea                                      |
|-------------------------|------------------------------------------------|
| Rule-Based              | If-then logic, keyword patterns                |
| ML-Driven               | Learns from labeled data                       |
| Generative              | Creates replies (no fixed responses)           |
| Hybrid                  | Combines rule-based + ML/generative            |
| Task-Oriented           | Focuses on completing a specific task          |
| Conversational Assistant| Engages in open-ended conversation             |



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# sklearn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix

# nltk
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
nltk.download('stopwords')
nltk.download('punkt_tab')

import string
import random
import difflib

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
df = pd.read_excel("/content/icthub_dataset.xlsx")
df.head()

,User Input,Category,Chatbot Response
0,Good morning,Greeting,Welcome to ICTHUB - For Digital Solutions! \n\...
1,Good afternoon,Greeting,Welcome to ICTHUB - For Digital Solutions! \n\...
2,Good evening,Greeting,Welcome to ICTHUB - For Digital Solutions! \n...
3,Hi there,Greeting,\n Welcome to ICTHUB - For Digital Solutions!...
4,Hello,Greeting,\n Welcome to ICTHUB - For Digital Solutions!...


In [ ]:
len(df['Category'].unique())

11

In [ ]:
df['Category'].value_counts()

,count
Category,
Asking for Internship Details,82
Thanking Message,69
General_Info_Services,68
AI Internship Details,66
data analysis internship details,64
Web Development internship details,64
Cyber Security Internship Details,64
flutter internship details,64
supply chain internship details,64


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 715 entries, 0 to 714
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   User Input        715 non-null    object
 1   Category          715 non-null    object
 2   Chatbot Response  715 non-null    object
dtypes: object(3)
memory usage: 16.9+ KB


In [ ]:
df.isnull().sum()

,0
User Input,0
Category,0
Chatbot Response,0


In [ ]:
df.duplicated().sum()

np.int64(2)

In [ ]:
df.drop_duplicates(inplace=True)

# Preprocessing

In [ ]:
def preprocessing_text(text):

  # transform the text to lowercase
  text = text.lower()

  # tokenziation using nltk
  text = word_tokenize(text)

  # remove special characthers
  text = [word for word in text if word.isalnum()]

  # remove stop words and punctaution
  stop_words = set(stopwords.words('english'))
  text = [word for word in text if word not in stop_words and word not in string.punctuation ]

  # stemming
  stemmer = PorterStemmer()
  text = [stemmer.stem(word) for word in text]

  return " ".join(text)

In [ ]:
df['processed_text'] = df['User Input'].apply(preprocessing_text)
df.head()

,User Input,Category,Chatbot Response,processed_text
0,Good morning,Greeting,Welcome to ICTHUB - For Digital Solutions! \n\...,good morn
1,Good afternoon,Greeting,Welcome to ICTHUB - For Digital Solutions! \n\...,good afternoon
2,Good evening,Greeting,Welcome to ICTHUB - For Digital Solutions! \n...,good even
3,Hi there,Greeting,\n Welcome to ICTHUB - For Digital Solutions!...,hi
4,Hello,Greeting,\n Welcome to ICTHUB - For Digital Solutions!...,hello


In [ ]:
X = df['processed_text']
y = df['Category']

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1, stratify=y)

tfidf_vectorizer = TfidfVectorizer()
X_train = tfidf_vectorizer.fit_transform(X_train)
X_test = tfidf_vectorizer.transform(X_test)

In [ ]:
model = LogisticRegression()
model.fit(X_train, y_train)

LogisticRegression()

In [ ]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy*100}")

Accuracy: 96.64804469273743


# chatbot

In [ ]:
know_words = list(tfidf_vectorizer.vocabulary_.keys())

def is_similar_to_know_word(processed_text,min_matches=1):
  words = processed_text.split()
  match_count = 0
  for word in words:
    close = difflib.get_close_matches(word, know_words, n=1, cutoff=0.75) # hellllo
    if close:
      match_count += 1
  return match_count >= min_matches

In [ ]:
def chatbot_response(user_input):

  # preprocessing user input
  user_input = preprocessing_text(user_input)

  # Reject if not similar to know words
  if not is_similar_to_know_word(user_input):
    return "I'm sorry, I don't understand. Can you please rephrase your question?"

  # and transform user input to number
  user_input = tfidf_vectorizer.transform([user_input])

  # predict user input
  predicted_category = model.predict(user_input)[0]
  predicted_category = label_encoder.inverse_transform([predicted_category])[0]

  responses = df.groupby("Category")["Chatbot Response"].apply(list).to_dict()
  response = random.choice(responses[predicted_category]).strip()

  return response

In [ ]:
while True:
  user_input = input("User: ")
  if user_input.lower() == 'exit':
    break
  else:
    print(chatbot_response(user_input))

User: exit


In [ ]:
# tflite
# joblib
# flask api
# streamlit

# GUI

In [ ]:
import gradio as gr
import random

def format_history(history):
    html = ""
    for role, msg in history:
        if role == "user":
            html += f"<div class='right-bubble'>{msg}</div>"
        else:
            html += f"<div class='left-bubble'>{msg}</div>"
    return html

def update_chat(user_input, history):
    bot_reply = chatbot_response(user_input)
    history.append(("user", user_input))
    history.append(("bot", bot_reply))
    return "", history, format_history(history)


with gr.Blocks(css="""
#chatbox {
    background-color: #e4e4e4;
    padding: 20px;
    border-radius: 15px;
    min-height: 400px;
    max-height: 400px;
    overflow-y: auto;
    font-family: sans-serif;
}
#chatbox .left-bubble {
    background-color: #F37021;
    color: white;
    padding: 10px 15px;
    border-radius: 15px;
    margin: 10px 0;
    max-width: 40%;
    text-align: left;
    animation: fadeIn 0.2s ease-in;
}
#chatbox .right-bubble {
    background-color: white;
    color: black;
    padding: 10px 15px;
    border-radius: 15px;
    margin: 10px 0;
    max-width: 20%;
    margin-left: auto;
    text-align: right;
    animation: fadeIn 0.2s ease-in;
}
@keyframes fadeIn {
    from { opacity: 0; transform: translateY(5px); }
    to { opacity: 1; transform: translateY(0); }
}
#inputbox textarea {
    border-radius: 25px;
    padding: 12px;
}
#newchat {
    border-radius: 25px;
    font-weight: bold;
    background-color: #F37021;
    color: white;
}

#contact-btn:hover {
    background-color: #1ebe5d;
}
#logo-left {
    margin-left: 0;
    margin-bottom: 20px;
}


""") as demo:

    gr.Image(
    value="icthub_logo.png",
    show_label=False,
    show_download_button=False,
    width=200,
    elem_id="logo-left"
)


    gr.HTML("""
    <div class='left-bubble'>
        <strong>Welcome to ICTHUB - For Digital Solutions!</strong><br><br>
        Hello there! I'm your virtual assistant.
        How can I assist you today?
    </div>
    """)

    chat_html = gr.HTML(elem_id="chatbox")
    chat_state = gr.State([])

    with gr.Row():
       with gr.Column(scale=1):
           clear_btn = gr.Button("New Chat", elem_id="newchat")
       with gr.Column(scale=4):
           user_input = gr.Textbox(placeholder="Type your message here...", elem_id="inputbox")

    gr.HTML("""
        <div style="text-align: center; margin-top: 15px;">
            <a href="https://wa.me/+201556427825" target="_blank">
                <button style="background-color: #25D366; color: white; padding: 10px 20px; border: none; border-radius: 20px; font-weight: bold;">
                    📞 Contact Us on WhatsApp
                </button>
            </a>
        </div>
    """)

    user_input.submit(update_chat, [user_input, chat_state], [user_input, chat_state, chat_html])
    clear_btn.click(lambda: ([], ""), None, [chat_state, chat_html], queue=False)

demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://898af0f83f21c9bf3b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
